<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/consequtive_day_purchases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date,col,lag,datediff,lit
from pyspark.sql.window import Window
# Create a SparkSession
spark = SparkSession.builder\
    .appName("CSV_Reader")\
    .getOrCreate()

In [ ]:
# Define the path to the CSV file
file_path = '/content/sample_data/purchases_100k_realistic.csv'

# Read the CSV file into a PySpark DataFrame
purchases_df = spark.read.csv(file_path, header=True, inferSchema=True)

# Display the first 5 rows of the DataFrame
purchases_df.show(5)

+-------+----------+--------+-------------------+
|user_id|product_id|quantity|      purchase_date|
+-------+----------+--------+-------------------+
|   8937|      4681|       5|2022-10-23 07:05:20|
|   3824|      2444|       3|2022-01-18 15:06:34|
|   3160|      4388|      19|2022-04-18 12:32:05|
|   6399|      4273|      13|2022-08-03 17:09:43|
|   7723|       783|       8|2022-09-21 08:48:55|
+-------+----------+--------+-------------------+
only showing top 5 rows


In [ ]:
# Print the schema to verify data types
purchases_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- purchase_date: timestamp (nullable = true)



In [ ]:
from pyspark.sql.functions import date_format

# Reformat the purchase_date column to 'YYYY-mm-DD'
purchases_df = purchases_df.withColumn("purchase_date_formatted", to_date(col("purchase_date"),"yyyy-MM-dd"))

# Display the first 5 rows with the new formatted column


In [ ]:
window=Window.partitionBy("user_id","product_id").orderBy("purchase_date_formatted")


In [ ]:
dup_drop_df=purchases_df.dropDuplicates(["user_id","product_id","purchase_date_formatted"])

In [ ]:
lagging_df=dup_drop_df.withColumn("previous_purchase_date",lag("purchase_date_formatted").over(window))

In [ ]:
lagging_df=lagging_df.drop("purchase_date")

In [ ]:
lagging_df=lagging_df.withColumnRenamed("purchase_date_formatted","purchase_date")

In [ ]:
lagging_df=lagging_df.withColumn("days_since_previous_purchase",datediff(col("purchase_date"),col("previous_purchase_date")))

In [ ]:
window=window.partitionBy("product_id",'user_id').orderBy("purchase_date_formatted")

In [ ]:
lagging_df_fillNA = lagging_df.fillna(0, subset=["days_since_previous_purchase"]).filter(col("days_since_previous_purchase")==1).groupBy("product_id","user_id","days_since_previous_purchase").count().filter(col("count")>=2)

In [ ]:
lagging_df_fillNA.show(100)

+----------+-------+----------------------------+-----+
|product_id|user_id|days_since_previous_purchase|count|
+----------+-------+----------------------------+-----+
|      2001|   1001|                           1|    2|
|      2002|   1002|                           1|    2|
|      2003|   1003|                           1|    2|
|      2004|   1004|                           1|    2|
|      2005|   1005|                           1|    2|
|      2006|   1006|                           1|    2|
|      2007|   1007|                           1|    2|
|      2008|   1008|                           1|    2|
|      2009|   1009|                           1|    2|
|      2010|   1010|                           1|    2|
+----------+-------+----------------------------+-----+

